In [ ]:
from em_analysis import EMAnalysis

sensor_id = 12345
path = rf'C:\Users\donna\VS Code\zaber-python\20hz\03 09 26_325mm2_EM\FUT'
sensor_type = int("3")

# Create analysis instance
analysis = EMAnalysis(path, sensor_id, sensor_type)

# Get results
result = analysis.save_data()

# from shear_analysis import ShearAnalysis
# shear_id = 67
# path = rf'C:\Users\emili\OneDrive\Documents\Projects\VenaVitals\zaber-python\67\11 24 25_50.27_SHEAR\FUT'
# shear_analysis = ShearAnalysis(path, shear_id)
# result = shear_analysis.run_full_analysis()


Run 1, CH 1: Noise Level 0.0017 -> Using Window 128
Run 1, CH 2: Noise Level 0.0019 -> Using Window 148
Run 1, CH 3: Noise Level 0.0020 -> Using Window 160
Run 1, CH 4: Noise Level 0.0021 -> Using Window 170
Run 1, CH 5: Noise Level 0.0014 -> Using Window 100
Run 1, CH 6: Noise Level 0.0014 -> Using Window 100
Run 1, CH 7: Noise Level 0.0017 -> Using Window 128
Run 1, CH 8: Noise Level 0.0026 -> Using Window 228


c:\Users\emili\OneDrive\Documents\Projects\VenaVitals\zaber-python\em_analysis.py:338: RuntimeWarning: divide by zero encountered in divide
  fir_dev_ij = np.diff(y_smooth) / np.diff(x_smooth)


Run 2, CH 1: Noise Level 0.0015 -> Using Window 105
Run 2, CH 2: Noise Level 0.0016 -> Using Window 122
Run 2, CH 3: Noise Level 0.0017 -> Using Window 129
Run 2, CH 4: Noise Level 0.0018 -> Using Window 140
Run 2, CH 5: Noise Level 0.0013 -> Using Window 100
Run 2, CH 6: Noise Level 0.0013 -> Using Window 100
Run 2, CH 7: Noise Level 0.0015 -> Using Window 115
Run 2, CH 8: Noise Level 0.0024 -> Using Window 206
Run 3, CH 1: Noise Level 0.0017 -> Using Window 136
Run 3, CH 2: Noise Level 0.0020 -> Using Window 158
Run 3, CH 3: Noise Level 0.0021 -> Using Window 171
Run 3, CH 4: Noise Level 0.0022 -> Using Window 183
Run 3, CH 5: Noise Level 0.0016 -> Using Window 121
Run 3, CH 6: Noise Level 0.0017 -> Using Window 132
Run 3, CH 7: Noise Level 0.0018 -> Using Window 143
Run 3, CH 8: Noise Level 0.0029 -> Using Window 255
Plotting Run 1, CH 1 with color [0.12156863 0.46666667 0.70588235 1.        ]
Plotting Run 1, CH 2 with color [0.12156863 0.46666667 0.70588235 1.        ]
Plotting Run

In [15]:
import h5py
import numpy as np
# for value in result['max_kpa']:
#     for value2 in value:
#         print(value2)
#     print('---')
print((result['zaber_x'][0][:20]))
# for value in result['mean_max_kpa']:
#     print(value)
with h5py.File(rf'C:\Users\emili\OneDrive\Documents\Projects\VenaVitals\zaber-python\12345\02 03 26_325mm2_EB_A\matlab_all_results.mat', 'r') as f:
    # List all variable names in the file
    print(list(f.keys()))
    ref = f['result']['zaber_x'][0][0]
    matlab_values = f[ref][:][0]

    print((matlab_values[:20]))

[0.00021935 0.00054683 0.00065599 0.00071057 0.00074332 0.00076515
 0.00078075 0.00079245 0.00080825 0.00083096 0.00085869 0.00088682
 0.00091063 0.00093104 0.00094873 0.0009642  0.00097786 0.00098999
 0.00101262 0.00104417]
['#refs#', 'result']
[0.00021935 0.00065599 0.00074332 0.00078075 0.00080825 0.00085869
 0.00091063 0.00094873 0.00097786 0.00101262 0.00108335 0.00115539
 0.0012159  0.00126745 0.00131505 0.00136276 0.00140593 0.00144416
 0.00147825 0.00150383]


In [ ]:
import h5py
import numpy as np
# Load the file
with h5py.File(rf'C:\Users\emili\OneDrive\Documents\Projects\VenaVitals\zaber-python\12345\02 03 26_325mm2_EB_A\matlab_results.mat', 'r') as f:
    # List all variable names in the file
    print(list(f.keys()))
    
    # Access a specific variable
    # Note: h5py often transposes the data compared to MATLAB
    ref = f['result']['test'][0][2]
    matlab_values = f[ref][:].T
    
    python_values = result['test'][2][0]
    python_values = python_values[:-1, :]

    # ref = f['result']['zaber_y'][0]
    # matlab_values = f[ref][:].T
    
    # python_values = result['zaber_y'][0]
    #python_values = python_values[:-1, :]
    

# 1. Ensure both are NumPy arrays
# (h5py already returns a numpy array for f[ref][:])
# (result['test'][0][0] might be a list, so we convert it)
py_arr = np.array(python_values) #[:-1]

# 2. Calculate the Residual (Element-wise difference)
residual = matlab_values - py_arr

# 3. View the results
print("Max Difference:", np.max(np.abs(residual)))
print("Mean Squared Error:", np.mean(residual**2))

import matplotlib.pyplot as plt
import numpy as np

# Assuming column 0 is Time, and columns 1-8 are the 8 channels
time_vec = matlab_values[:, 0]

# Create a 4x2 grid
fig, axes = plt.subplots(4, 2, figsize=(15, 12), sharex=True)
axes = axes.flatten()

for i in range(8):
    ax = axes[i]
    
    # Column index is i + 1 because column 0 is time
    col_idx = i + 1
    
    # Plot using the actual time vector for the X-axis
    ax.plot(time_vec, matlab_values[:, col_idx], label='MATLAB', color='blue', alpha=0.7)
    ax.plot(time_vec, py_arr[:, col_idx], label='Python', color='orange', linestyle='--', alpha=0.8)
    
    ax.set_title(f'Channel {i+1}')
    ax.set_ylabel('Cap (pF)')
    ax.grid(True, alpha=0.3)
    
    if i == 0:
        ax.legend()

# Set X-label for the bottom-most plots
axes[6].set_xlabel('Time (s)')
axes[7].set_xlabel('Time (s)')

plt.suptitle('8-Channel Sync Check (Time-Based X-Axis)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

['#refs#', 'result']
25387
25386


ValueError: operands could not be broadcast together with shapes (25386,9) (25387,9) 

In [2]:
for value in result['max_kpa']:
    for value2 in value:
        print(value2)
    print('---')

# for value in result['std_max_ps']:
#     print(value)


7.830086697532192
6.217285448725904
5.861602156867043
5.887858215658926
5.5672336350905445
6.096575001904743
7.180158366222451
8.409492100666508
---
7.320366266276052
6.500253507132155
5.473351990951808
5.692554418654912
4.96900063457162
6.185120350975541
6.916665357934515
8.373767932169102
---
6.54467829659655
6.2685828123958
5.643582329279693
5.526265781338013
5.1076068491312885
5.779656389746903
6.492327791045018
8.097622356654156
---


In [9]:
# The v=5 parameter means it starts extracting from column 5
# But maybe the actual CAP data (change values) are in different columns?
# Let me check the _correct_ch_order logic

print("Channel order correction:")
print(f"sensor_type: {analysis.sensor_type}")
print(f"ch_order: {analysis.ch_order}")
print(f"v (version): {analysis.v}")

cols = (
    list(range(0, 5)) +
    list(analysis.ch_order + analysis.v - 1) +
    list(range(13, 16))
)
print(f"Selected column indices: {cols}")
print(f"So the reordered dataframe will use:")
print(f"  Cols 0-4: original cols {list(range(0, 5))}")
print(f"  Cols 5-12 (CAP): original cols {list(analysis.ch_order + analysis.v - 1)}")
print(f"  Cols 13-15: original cols {list(range(13, 16))}")

print(f"\nThen in _interp_cap(), it extracts columns starting at v={analysis.v}:")
print(f"  col_idx = j + v, so j=0..7 gives col_idx = 5..12")
print(f"  These should be the 8 CAP channels")

# Let's compare what MATLAB might be using
print(f"\nIn MATLAB code from user:")
print(f"  zaber_y{{i,j}} = smooth(y(st_pt:end),100);")
print(f"  This reads the y data for channel j")
print(f"  User said raw values in MATLAB are ~2.5x higher than Python")

# Check the ratio
print(f"\nActual CAP value ratio check:")
print(f"  Column 5 (absolute): 24.37")
print(f"  Column 13 (change?): -0.0183")
print(f"  Column 14 (change?): 0.0029")
print(f"  Column 15 (change?): -0.9975")
print(f"\nRatio of col5 to col15: {24.37 / 0.9975:.1f}x (not 2.5x though)")


Channel order correction:
sensor_type: 3
ch_order: [1 2 3 4 8 7 6 5]
v (version): 5
Selected column indices: [0, 1, 2, 3, 4, np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(12), np.int64(11), np.int64(10), np.int64(9), 13, 14, 15]
So the reordered dataframe will use:
  Cols 0-4: original cols [0, 1, 2, 3, 4]
  Cols 5-12 (CAP): original cols [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(12), np.int64(11), np.int64(10), np.int64(9)]
  Cols 13-15: original cols [13, 14, 15]

Then in _interp_cap(), it extracts columns starting at v=5:
  col_idx = j + v, so j=0..7 gives col_idx = 5..12
  These should be the 8 CAP channels

In MATLAB code from user:
  zaber_y{i,j} = smooth(y(st_pt:end),100);
  This reads the y data for channel j
  User said raw values in MATLAB are ~2.5x higher than Python

Actual CAP value ratio check:
  Column 5 (absolute): 24.37
  Column 13 (change?): -0.0183
  Column 14 (change?): 0.0029
  Column 15 (change?): -0.9975

Ratio of col5 to col15:

In [4]:
# Check if the range constraint is working
from scipy.signal import savgol_filter
import numpy as np

i = 0
j = 0

k = np.where(analysis.test[i][1][:,1] - analysis.start_force > 0)[0][0]
f = np.where(analysis.test[i][1][:,1] - analysis.end_force > 0)[0][0]

x = analysis.test[i][1][k:f, 1]  
y = analysis.test[i][0][k:f, j+1]

st_pt = np.where(x-0 > 0)[0][0]

x_smooth = savgol_filter(x[st_pt:], 101, 2)
y_smooth = savgol_filter(y[st_pt:], 101, 2)

fir_dev = np.diff(y_smooth) / np.diff(x_smooth)
fir_dev[(fir_dev > 1) | (fir_dev < 0)] = 0

# Check range constraint
search_min = 5
search_max = 40
valid_mask = (x_smooth >= search_min) & (x_smooth <= search_max)
valid_indices = np.where(valid_mask)[0]

print(f"x_smooth shape: {x_smooth.shape}")
print(f"fir_dev shape: {fir_dev.shape}")
print(f"Valid force indices: {len(valid_indices)} points")
print(f"First valid index: {valid_indices[0] if len(valid_indices) > 0 else 'None'}")
print(f"Last valid index: {valid_indices[-1] if len(valid_indices) > 0 else 'None'}")
if len(valid_indices) > 0:
    print(f"x_smooth[valid_indices[0:5]]: {x_smooth[valid_indices[0:5]]}")
    print(f"fir_dev[valid_indices[0:5]]: {fir_dev[valid_indices[0:5]]}")

    # Max in valid range
    valid_deriv = fir_dev[valid_indices]
    max_valid_idx_local = np.argmax(valid_deriv)
    max_valid_idx = valid_indices[max_valid_idx_local]
    print(f"\nMax in range: {fir_dev[max_valid_idx]:.6f}")
    print(f"Force at max in range: {x_smooth[max_valid_idx]:.6f}")
    print(f"MATLAB expected: ~12.5 kPa")

Number of peaks found: 1
Peak 0: index=20128, value=0.204591, force=22.473888

Peaks in valid range (5-40 kPa): 1
Valid peak 0: index=20128, value=0.204591, force=22.473888


Number of peaks found: 1
Peak 0: index=20128, value=0.204591, force=22.473888

Peaks in valid range (5-40 kPa): 1
Valid peak 0: index=20128, value=0.204591, force=22.473888


C:\Users\emili\AppData\Local\Temp\ipykernel_62132\993273915.py:20: RuntimeWarning: divide by zero encountered in divide
  fir_dev = np.diff(y_smooth) / np.diff(x_smooth)
